# T2 — RAG avec LangChain & Mistral · Jour 3

Un LLM est entraîné sur un corpus **figé** : il ignore vos données privées et les
informations récentes. Le **RAG** (*Retrieval-Augmented Generation*) contourne cette
limite en **récupérant** des documents pertinents pour **ancrer** la génération.

Le corpus de démonstration est un billet de blog public sur les agents LLM (Lilian Weng),
choisi car dense et bien structuré. Il se remplace par vos propres documents dans l'atelier.

## 🎯 Objectifs

À la fin de ce notebook, vous savez :

1. construire le **cycle RAG de base** : *indexing → retrieval → generation* en LCEL sur Mistral ;
2. comprendre pourquoi la recherche par similarité **échoue** sur certaines questions ;
3. mettre en œuvre trois techniques de **traduction de requête** (*query translation*) :
   **Multi-Query**, **RAG-Fusion** (RRF) et **HyDE** ;
4. comparer les réponses obtenues et savoir **quand** chaque technique aide.

## 🗺️ Plan

| Partie | Sujet |
|---|---|
| 1 | Indexing — charger, découper, vectoriser |
| 2 | Retrieval — rechercher par similarité |
| 3 | Generation — la chaîne RAG minimale |
| 4a | Multi-Query — plusieurs angles d'une question |
| 4b | RAG-Fusion — fusionner les classements (RRF) |
| 4c | HyDE — document hypothétique |
| 5 | Comparaison des approches |

## 📖 Glossaire express

- **Chunk** — fragment d'un document ; `chunk_size` et `chunk_overlap` en règlent la taille.
- **Embedding** — vecteur dense qui encode le *sens* d'un texte (`mistral-embed`).
- **VectorStore** — base optimisée pour la recherche par similarité (ici **Chroma**, en mémoire).
- **Retriever** — objet qui renvoie les `k` chunks les plus proches d'une requête.
- **LCEL** — *LangChain Expression Language* : composer des étapes avec l'opérateur `|`.
- **Query translation** — reformuler la question pour améliorer la récupération.
- **RRF** — *Reciprocal Rank Fusion* : fusionner plusieurs classements en un seul score.
- **HyDE** — *Hypothetical Document Embeddings* : chercher avec une **réponse** hypothétique plutôt qu'avec la question.

## 🛠️ Prérequis — dépendances

Ce notebook ajoute des paquets par rapport au Jour 2 (`langchain-community` pour le
loader web, `langchain-chroma` + `chromadb` pour l'index, `beautifulsoup4` pour parser
le HTML). `langchain-mistralai` est déjà présent depuis le Jour 2.

```powershell
# Depuis J3_Avancé (réutilise ou crée un .venv dédié)
uv pip install langchain-community langchain-chroma chromadb beautifulsoup4
```

> **Piège.** Certains serveurs compatibles Mistral n'exposent pas la route
> `/v1/embeddings`. Si l'appel aux embeddings échoue, vérifiez auprès de votre
> administrateur que le modèle `mistral-embed` est bien servi.

## Configuration de l'environnement

On lit `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL` depuis l'environnement (ou un `.env`
local). On **normalise** l'URL pour qu'elle se termine par exactement un `/v1`, quel
que soit le format fourni. Rien de secret n'est affiché.

In [ ]:
import os, sys
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# WebBaseLoader réclame un User-Agent : on en fixe un pour éviter un avertissement.
os.environ.setdefault("USER_AGENT", "formation-mistral-j3")

# Normalise l'URL du serveur : se termine par exactement un /v1.
_base = os.environ["MISTRAL_SERVER_URL"].rstrip("/")
MISTRAL_ENDPOINT = _base if _base.endswith("/v1") else _base + "/v1"

# Vérification masquée des variables (via util/env_utils.py, comme au Jour 2).
# Le notebook est dans J3_Avancé/T2_RAG/ → la racine est deux niveaux au-dessus.
_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, _root)
from util.env_utils import doublecheck_env

doublecheck_env(os.path.join(_root, ".env"))
print("Endpoint Mistral :", MISTRAL_ENDPOINT.replace(_base.split("//")[-1].split("/")[0], "****"))

In [ ]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

MODEL = "mistral-medium-latest"

# LLM de génération — temperature=0 pour des réponses reproductibles.
llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    endpoint=MISTRAL_ENDPOINT,
)

# Modèle d'embeddings — même serveur, même endpoint.
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    endpoint=MISTRAL_ENDPOINT,
)

print(f"✅ LLM : {MODEL} · embeddings : mistral-embed")

---
## Partie 1 — Indexing

**Pourquoi ?** Le modèle ne peut pas lire toute une base à chaque question. On la
**pré-traite une fois** : on charge les documents, on les **découpe** en chunks, on
calcule leurs **embeddings**, et on les range dans un **index vectoriel**. La question
sera ensuite comparée à ces vecteurs.

On charge un seul article (le billet de Lilian Weng), en ne gardant que le contenu utile
du HTML grâce à un `SoupStrainer`.

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

print(f"{len(docs)} document(s) chargé(s) · {len(docs[0].page_content)} caractères")

On découpe en chunks de ~1000 caractères avec 200 de chevauchement. Le chevauchement
évite de **couper une idée en deux** entre deux chunks voisins.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits = text_splitter.split_documents(docs)

print(f"{len(splits)} chunks")
print("Exemple :", splits[0].page_content[:200], "...")

On vectorise les chunks avec `mistral-embed` et on les indexe dans **Chroma** (en mémoire).
Cette cellule appelle le serveur : elle prend quelques secondes.

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Index prêt · retriever configuré (k=4)")

> **À retenir.** L'indexing se fait **une fois**. Les deux leviers de qualité sont la
> **taille des chunks** et le **modèle d'embeddings**. Un chunk trop grand noie
> l'information ; trop petit, il perd le contexte.

---
## Partie 2 — Retrieval

**Pourquoi ?** Face à une question, on récupère les `k` chunks dont l'embedding est le
plus **proche** (similarité cosinus) de celui de la question. C'est cette étape qui
**ancre** le modèle dans vos données.

In [ ]:
question = "Qu'est-ce que la décomposition de tâches pour un agent LLM ?"

docs_retrouves = retriever.invoke(question)

print(f"{len(docs_retrouves)} chunks récupérés\n")
print(docs_retrouves[0].page_content[:400], "...")

> **À retenir.** La recherche est **sémantique**, pas par mots-clés : une question et un
> passage peuvent matcher sans partager le moindre mot. Mais si la question est mal
> formulée, on récupère les mauvais chunks — c'est le problème que règle la Partie 4.

---
## Partie 3 — Generation

**Pourquoi ?** On assemble maintenant la chaîne complète en **LCEL** : les chunks
récupérés sont injectés dans le **contexte** d'un prompt, que le LLM Mistral utilise
pour répondre — **uniquement** à partir de ce contexte.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """Réponds à la question en te basant UNIQUEMENT sur le contexte suivant.
Si le contexte ne contient pas la réponse, dis-le honnêtement.

Contexte :
{context}

Question : {question}
"""
prompt = ChatPromptTemplate.from_template(template)


def format_docs(docs):
    """Concatène le texte des chunks pour l'insérer dans le prompt."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

reponse = rag_chain.invoke(question)
print(reponse)

> **À retenir.** Le cœur du RAG tient en une ligne LCEL :
> `{"context": retriever, "question": ...} | prompt | llm | parser`.
> Tout le reste (Partie 4) consiste à **améliorer ce qui entre dans `context`**.

---
## Partie 4 — Query Translation

**Le problème.** La récupération repose sur la proximité entre l'embedding de la
**question** et ceux des **chunks**. Si l'utilisateur formule mal sa question, ou avec
un vocabulaire différent de celui des documents, on récupère des chunks hors-sujet — et
le meilleur LLM du monde répondra mal sur un mauvais contexte.

**L'idée.** Transformer la question **avant** la récupération. On voit trois techniques :

| Technique | Idée en une phrase |
|---|---|
| **Multi-Query** | poser la question sous **plusieurs angles**, puis unir les résultats |
| **RAG-Fusion** | idem, mais **fusionner les classements** avec un score (RRF) |
| **HyDE** | chercher avec une **réponse hypothétique** au lieu de la question |

### 4a — Multi-Query

**Pourquoi ?** Une seule formulation ne capture qu'un angle. On demande au LLM de
**reformuler** la question de plusieurs façons, on récupère pour **chacune**, puis on
prend l'**union** des chunks uniques. On élargit ainsi le filet.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template_multiquery = """Tu es un assistant de recherche. Génère CINQ reformulations
différentes de la question ci-dessous, pour interroger une base vectorielle sous
plusieurs angles. Objectif : contourner les limites de la recherche par similarité.
Donne les cinq questions, une par ligne, sans numérotation.

Question originale : {question}"""

prompt_multiquery = ChatPromptTemplate.from_template(template_multiquery)

generate_queries = (
    prompt_multiquery
    | llm
    | StrOutputParser()
    | (lambda texte: [q.strip() for q in texte.split("\n") if q.strip()])
)

# Aperçu des reformulations générées
for q in generate_queries.invoke({"question": question}):
    print("•", q)

In [ ]:
from langchain_core.load import dumps, loads


def union_unique(documents: list[list]):
    """Aplati les listes de chunks et déduplique (sérialisation → set → désérialisation)."""
    aplati = [dumps(doc) for sous_liste in documents for doc in sous_liste]
    uniques = list(set(aplati))
    return [loads(doc) for doc in uniques]


# generate_queries -> une liste de questions ; retriever.map() -> une liste de listes de chunks
retrieval_multiquery = generate_queries | retriever.map() | union_unique

chunks = retrieval_multiquery.invoke({"question": question})
print(f"{len(chunks)} chunks uniques récupérés (contre 4 avec la requête simple)")

In [ ]:
from operator import itemgetter

rag_multiquery = (
    {"context": retrieval_multiquery | format_docs, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_multiquery.invoke({"question": question}))

> **Note honnête.** Multi-Query **multiplie les appels** (une génération + N récupérations).
> Le gain de rappel se paie en latence et en tokens. À réserver aux questions où la
> requête simple échoue.

### 4b — RAG-Fusion

**Pourquoi ?** Multi-Query prend l'union « à plat » : un chunk qui ressort **en tête pour
plusieurs reformulations** est traité comme n'importe quel autre. **RAG-Fusion** corrige
cela avec le **Reciprocal Rank Fusion (RRF)** : chaque chunk gagne un score
`1 / (rang + k)` dans chaque liste, et les scores s'**additionnent**. Un chunk souvent
bien classé remonte en tête.

In [ ]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(resultats: list[list], k: int = 60):
    """Fusionne plusieurs classements de chunks en un seul (RRF)."""
    scores = {}
    for chunks in resultats:
        for rang, doc in enumerate(chunks):
            cle = dumps(doc)
            scores[cle] = scores.get(cle, 0.0) + 1.0 / (rang + k)
    classes = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [(loads(cle), score) for cle, score in classes]


retrieval_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion

classement = retrieval_fusion.invoke({"question": question})
print(f"{len(classement)} chunks classés par score RRF\n")
for doc, score in classement[:3]:
    print(f"[{score:.4f}] {doc.page_content[:90]} ...")

In [ ]:
def format_fusion(resultats_scores):
    """Ne garde que le texte des chunks (on ignore le score pour le prompt)."""
    return "\n\n".join(doc.page_content for doc, _score in resultats_scores)


rag_fusion = (
    {"context": retrieval_fusion | format_fusion, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_fusion.invoke({"question": question}))

> **À retenir.** `k=60` est la valeur usuelle du RRF (issue de l'article d'origine) : elle
> **atténue** le poids des tout premiers rangs pour ne pas laisser une seule liste dominer.

### 4c — HyDE (Hypothetical Document Embeddings)

**Pourquoi ?** Une **question** courte et une **réponse** rédigée n'ont pas la même
« forme » sémantique — or l'index contient des **passages de type réponse**. HyDE demande
d'abord au LLM d'**inventer** un passage qui répondrait à la question, puis récupère avec
l'embedding de ce **passage hypothétique**. On compare des documents à des documents.

> **Piège.** Le passage inventé peut contenir des **faits faux** (hallucination). Ce n'est
> pas grave : il ne sert qu'à **guider la recherche**, jamais à répondre. La réponse finale
> reste ancrée dans les chunks **réels** récupérés.

In [ ]:
template_hyde = """Rédige un court paragraphe de documentation technique qui répondrait
à la question suivante. Sois factuel et concis (4 à 5 phrases).

Question : {question}
Paragraphe :"""

prompt_hyde = ChatPromptTemplate.from_template(template_hyde)

generer_doc_hypothetique = prompt_hyde | llm | StrOutputParser()

doc_hypothetique = generer_doc_hypothetique.invoke({"question": question})
print("Document hypothétique généré :\n")
print(doc_hypothetique)

In [ ]:
# On récupère avec le document hypothétique au lieu de la question brute.
retrieval_hyde = generer_doc_hypothetique | retriever

rag_hyde = (
    {"context": retrieval_hyde | format_docs, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_hyde.invoke({"question": question}))

> **À retenir.** HyDE brille quand la question est **courte ou vague** et que les documents
> sont **denses**. Il ajoute un appel LLM en amont — même arbitrage latence/qualité que les
> autres techniques.

---
## Partie 5 — Comparaison des approches

Posons **la même question** aux quatre chaînes et regardons les réponses côte à côte.
Il n'y a pas de gagnant universel : le bon choix dépend du corpus et des questions.

In [ ]:
chaines = {
    "RAG simple":  rag_chain,
    "Multi-Query": rag_multiquery,
    "RAG-Fusion":  rag_fusion,
    "HyDE":        rag_hyde,
}

q_test = "Quelles sont les composantes d'un système agent piloté par un LLM ?"

for nom, chaine in chaines.items():
    # rag_chain prend une chaîne ; les autres prennent un dict {"question": ...}
    entree = q_test if nom == "RAG simple" else {"question": q_test}
    reponse = chaine.invoke(entree)
    print(f"\n===== {nom} =====")
    print(reponse[:600])

### 🧭 Quand utiliser quoi ?

| Situation | Technique conseillée |
|---|---|
| Questions bien formulées, corpus homogène | **RAG simple** (le moins cher) |
| Questions ambiguës ou à multiples facettes | **Multi-Query** / **RAG-Fusion** |
| Plusieurs reformulations pointent les mêmes chunks | **RAG-Fusion** (RRF) |
| Questions courtes/vagues, documents denses | **HyDE** |

> **Note honnête.** Toutes ces techniques **améliorent la récupération**, pas la
> génération : si l'information **n'est pas dans le corpus**, aucune ne l'inventera
> (et c'est tant mieux). Mesurez toujours avec un jeu de questions/réponses de référence
> (voir l'atelier et RAGAS).

## 🔭 Pour aller plus loin

- **Routing** — router la question vers le bon index/outil.
- **Query Construction** — traduire en filtres de métadonnées / SQL.
- **Indexing avancé** — multi-représentation, RAPTOR, ColBERT.
- **Retrieval robuste** — re-ranking, **CRAG**, **Self-RAG**, contexte long.

## 📚 Ressources

- [LangChain — Tutoriel RAG](https://python.langchain.com/docs/tutorials/rag/)
- [LangChain — VectorStores](https://python.langchain.com/docs/concepts/vectorstores/)
- [LangChain — Retrievers](https://python.langchain.com/docs/concepts/retrievers/)
- [Mistral — Embeddings](https://docs.mistral.ai/capabilities/embeddings/)
- [RRF — article d'origine (Cormack et al.)](https://plg.uwaterloo.ca/~gvcormas/cormacksigir09-rrf.pdf)
- [HyDE — Precise Zero-Shot Dense Retrieval](https://arxiv.org/abs/2212.10496)